In [1]:
import pandas as pd
import matplotlib as plt
import numpy as np
import json, lz4.frame
import sys
import os
from pathlib import Path  
PATH = "C:\\Users\\omran\\code\\PokemonRL\\data\\data\\gen9ou\\2022\\11\\" 
# view one example
dir = PATH + "smogtours-gen9ou-663321_Unrated_pluck78484_vs_drifloon53330_11-30-2022_WIN.json.lz4"
with lz4.frame.open(dir, "rb") as f:
    data = json.loads(f.read().decode("utf-8"))
states, actions = data["states"], data["actions"]

In [ ]:
#helper functions
def sortNames(state, action) -> tuple:
    moves = [m['name'] for m in states[0]["player_active_pokemon"]["moves"]]
    moves.sort()
    
    swaps = [s['base_species'] for s in states[0]['available_switches']]
    swaps.sort()

    return (moves, swaps)

sortNames(states[0], actions[0])

(['energyball', 'mortalspin', 'sludgewave', 'spikes'],
 ['chienpao', 'gholdengo', 'ironvaliant', 'rotom', 'tinglu'])

In [18]:
states[0]
# print(actions[0])

{'format': 'gen9ou',
 'player_active_pokemon': {'name': 'glimmora',
  'hp_pct': 1.0,
  'types': 'poison rock',
  'item': 'focussash',
  'ability': 'toxicdebris',
  'lvl': 100,
  'status': 'nostatus',
  'effect': 'noeffect',
  'moves': [{'name': 'spikes',
    'move_type': 'ground',
    'category': 'status',
    'base_power': 0,
    'accuracy': 1.0,
    'priority': 0,
    'current_pp': 32,
    'max_pp': 32},
   {'name': 'mortalspin',
    'move_type': 'poison',
    'category': 'physical',
    'base_power': 30,
    'accuracy': 1.0,
    'priority': 0,
    'current_pp': 24,
    'max_pp': 24},
   {'name': 'energyball',
    'move_type': 'grass',
    'category': 'special',
    'base_power': 90,
    'accuracy': 1.0,
    'priority': 0,
    'current_pp': 16,
    'max_pp': 16},
   {'name': 'sludgewave',
    'move_type': 'poison',
    'category': 'special',
    'base_power': 95,
    'accuracy': 1.0,
    'priority': 0,
    'current_pp': 16,
    'max_pp': 16}],
  'atk_boost': 0,
  'spa_boost': 0,
  'd

In [27]:
#embedding from scratch
import torch
import random
import torch.nn as nn

class SharedEmbedding(nn.Module):
    def __init__(self, vocab_size=2540, embed_dim=3):
        super().__init__()

        #embeddings are basically ways to represent non-numerical data as numbers
        #they are vectors of whatever embed_dim is

        #they start as random values
        #but through backprop, then we can adjust them and figure out should be closer to each other 
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        
    #returns the idx
    def forward(self, idx):
        return self.embeddings(idx)
        


In [28]:
class PokemonEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, numeric_dim):
        super().__init__()
        self.embed = SharedEmbedding(vocab_size, embed_dim)

        self.output_dimensions = 11 *embed_dim + numeric_dim

    def forward(self, species_idx, item_idx, ability_idx, status_idx,
                tera_idx, type_idxs, move_idxs, numeric_feats):
        
        #access the vectors for each index from the embeddings
        species_vector = self.embed(species_idx)
        item_vector = self.embed(item_idx)
        ability_vector = self.embed(ability_idx)
        status_vector = self.embed(status_idx)
        tera_vector = self.embed(tera_idx)
        
        #flatten the 3D tensors into 2D since types are 2 long and moves are 4
        type_vector = self.embed(type_idxs)    
        type_vector = type_vector.reshape(type_vector.shape[0],-1)

        move_vector = self.embed(move_idxs)
        move_vector = move_vector.reshape(move_vector.shape[0],-1)

        out = torch.cat([species_vector,item_vector,ability_vector,status_vector,tera_vector, type_vector, move_vector, numeric_feats],
                        dim = -1)

        return out

In [31]:
class FieldEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, numeric_dim):
        super().__init__()
        self.embed = SharedEmbedding(vocab_size, embed_dim)

        #weather, player conditions, opp conditions, battle_field
        self.output_dimensions = 4 *embed_dim + numeric_dim

    def forward(self, weather_idx, player_conditions_idx, opp_conditions_idx, battle_field_idx, numeric_feats):
        
        #access the vectors for each index from the embeddings
        weather_vector = self.embed(weather_idx)
        player_conditions_vector = self.embed(player_conditions_idx)
        opp_conditions_vector = self.embed(opp_conditions_idx)
        battle_field_vector = self.embed(battle_field_idx)
        

        out = torch.cat([weather_vector,player_conditions_vector,opp_conditions_vector,battle_field_vector, numeric_feats],
                        dim = -1)

        return out

In [ ]:
class BattleStateEncoder(nn.Module):
    def __init__(self, field_encoder, pokemon_encoder, num_bench=5, num_opp_revealed=5):
        super().__init__()
        self.field_encoder = field_encoder
        self.pokemon_encoder = pokemon_encoder
        self.num_bench = num_bench
        self.num_opp_revealed = num_opp_revealed

        pkmn_dim = pokemon_encoder.output_dimensions
        self.output_dimensions = (
            field_encoder.output_dimensions
            + pkmn_dim                        # my active
            + pkmn_dim * num_bench            # my bench
            + pkmn_dim                        # opp active
            + pkmn_dim * num_opp_revealed     # opp revealed
        )

    def _encode_group(self, feats, num_slots):
        batch_size = feats["species_idx"].shape[0]
        flat_feats = {k: v.reshape(batch_size * num_slots, *v.shape[2:]) for k, v in feats.items()}
        out = self.pokemon_encoder(**flat_feats)                 # (batch*num_slots, pkmn_dim)
        return out.reshape(batch_size, num_slots * out.shape[-1])  # (batch, num_slots*pkmn_dim)

    def forward(self, field_feats, mine_active, mine_bench, opp_active, opp_revealed):
        field_enc = self.field_encoder(*field_feats)

        active_enc = self.pokemon_encoder(**mine_active)
        opp_active_enc = self.pokemon_encoder(**opp_active)

        bench_enc = self._encode_group(mine_bench, self.num_bench)
        opp_revealed_enc = self._encode_group(opp_revealed, self.num_opp_revealed)

        return torch.cat(
            [field_enc, active_enc, bench_enc, opp_active_enc, opp_revealed_enc],
            dim=-1,
        )

In [ ]:
class BCPolicy(nn.Module):
    def __init(self, state_encoder, action_dim=9, hidden_dim=256):
        super().init()
        self.state_encoder = state_encoder
        self.head = 

In [35]:
with open("DefaultObservationSpace-v1.json", "r") as file:
    vocab = json.load(file)

#example
mon = states[0]["player_active_pokemon"]

species_idx = torch.tensor([vocab[mon['name']]])          
item_idx    = torch.tensor([vocab[mon['item']]])          
ability_idx = torch.tensor([vocab[mon['ability']]])        
status_idx  = torch.tensor([vocab[mon['status']]])         
tera_idx    = torch.tensor([vocab[mon['tera_type']]])      

types = mon["types"].split()
type_idxs = torch.tensor([[vocab[t] for t in types]])    


moves = [m['name'] for m in mon['moves']]
moves_idxs = torch.tensor([[vocab[m] for m in moves]])    


numeric_feats = torch.tensor([[
    mon['hp_pct'],
    mon['atk_boost'] / 6.0, mon['spa_boost'] / 6.0,
    mon['def_boost'] / 6.0, mon['spd_boost'] / 6.0, mon['spe_boost'] / 6.0,
    mon['base_atk'] / 200.0, mon['base_spa'] / 200.0, mon['base_def'] / 200.0,
    mon['base_spd'] / 200.0, mon['base_spe'] / 200.0, mon['base_hp'] / 200.0,
]], dtype=torch.float32)  # shape (1, 12)

encoder = PokemonEncoder(vocab_size=2540, embed_dim=3, numeric_dim=12)
out = encoder(species_idx, item_idx, ability_idx, status_idx, tera_idx,
              type_idxs, moves_idxs, numeric_feats)

print(out.shape)


torch.Size([1, 45])


In [ ]:
class BCPolicy(nn.Module):
    def __init__(self, state_encoder, action_dim=9, hidden_dim=256):
        super().__init__()
        self.state_encoder = state_encoder
        self.head = nn.Sequential(
            nn.Linear(state_encoder.output_dimensions, hidden_dim), # y = w^T * x + b
            nn.ReLU(), # max(x,0)
            nn.Linear(hidden_dim, action_dim),
        )
    def forward(self, field_feats, mine_active, mine_bench, opp_active, opp_revealed, action_mask):
        # forward pass into the Neural Network
        state = self.state_encoder(field_feats, mine_active, mine_bench, opp_active, opp_revealed)
        logits = self.head(state)
        logits = logits.masked_fill(~action_mask, float("-inf")) #for any action that is illegal(switch to fainted mon, etc.)
        return logits

In [33]:
states[0]["player_active_pokemon"]

{'name': 'glimmora',
 'hp_pct': 1.0,
 'types': 'poison rock',
 'item': 'focussash',
 'ability': 'toxicdebris',
 'lvl': 100,
 'status': 'nostatus',
 'effect': 'noeffect',
 'moves': [{'name': 'spikes',
   'move_type': 'ground',
   'category': 'status',
   'base_power': 0,
   'accuracy': 1.0,
   'priority': 0,
   'current_pp': 32,
   'max_pp': 32},
  {'name': 'mortalspin',
   'move_type': 'poison',
   'category': 'physical',
   'base_power': 30,
   'accuracy': 1.0,
   'priority': 0,
   'current_pp': 24,
   'max_pp': 24},
  {'name': 'energyball',
   'move_type': 'grass',
   'category': 'special',
   'base_power': 90,
   'accuracy': 1.0,
   'priority': 0,
   'current_pp': 16,
   'max_pp': 16},
  {'name': 'sludgewave',
   'move_type': 'poison',
   'category': 'special',
   'base_power': 95,
   'accuracy': 1.0,
   'priority': 0,
   'current_pp': 16,
   'max_pp': 16}],
 'atk_boost': 0,
 'spa_boost': 0,
 'def_boost': 0,
 'spd_boost': 0,
 'spe_boost': 0,
 'accuracy_boost': 0,
 'evasion_boost': 0